# 03-5. 記述統計 — 動かして確かめる

📖 解説: [`../05_descriptive_stats.md`](../05_descriptive_stats.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. 平均 vs 中央値 — 外れ値が 1 つあるだけで何が起きるか
2. 【対話】社長の年収を動かして平均を壊す
3. ばらつきの指標と `ddof` の罠
4. 五数要約と箱ひげ図
5. 歪度・尖度 — 分布の形を数値にする
6. ⚠️ アンスコムの四重奏 — 要約統計量が同じでも図は別物
7. 欠損値の扱い

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (05_descriptive_stats.md)](../05_descriptive_stats.md)

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

rng = np.random.default_rng(42)

## 1. 平均 vs 中央値 — 社長が 1 人いるだけで

社員 10 人の年収 (万円)。最後の 1 人が創業社長です。

In [ ]:
# 年収データ (単位: 万円), shape: (10,)
income: np.ndarray = np.array([300, 320, 350, 360, 380, 400, 420, 450, 500, 5000])

print(f'平均値 : {np.mean(income):>7.1f} 万円   ← 社長に引っ張られる')
print(f'中央値 : {np.median(income):>7.1f} 万円   ← 「真ん中の人」の実感')
print()
print('「平均年収 848 万円の会社」と聞いて想像する姿と、実態は違う。')
print(f'実際、10 人中 {np.sum(income < np.mean(income))} 人が平均以下です。')

## 2. 【対話】社長の年収を動かしてみる

スライダーで社長の年収だけを変えます。

**注目点**: 平均は青天井に動くのに、**中央値はまったく動きません**。
これが「中央値は外れ値に強い（頑健）」ということです。

In [ ]:
def compare_center(ceo_income: float = 5000) -> None:
    """社長の年収を変えたときの平均と中央値の動きを描く.

    Args:
        ceo_income: 社長の年収 (単位: 万円)
    """
    base = np.array([300, 320, 350, 360, 380, 400, 420, 450, 500])  # shape: (9,)
    data = np.append(base, ceo_income)                              # shape: (10,)

    mean_v, median_v = data.mean(), np.median(data)

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.scatter(base, np.zeros_like(base), s=80, alpha=0.7, label='社員 9 人')
    ax.scatter([ceo_income], [0], s=160, c='orange', marker='D', label='社長')
    ax.axvline(mean_v, c='r', ls='--', lw=2, label=f'平均 {mean_v:.0f}')
    ax.axvline(median_v, c='g', ls='-', lw=2, label=f'中央値 {median_v:.0f}')
    ax.set_xlim(0, 11000)
    ax.set_yticks([])
    ax.set_xlabel('年収 (万円)')
    ax.set_title(f'社長 {ceo_income:.0f} 万円 → 平均 {mean_v:.0f} / 中央値 {median_v:.0f}')
    ax.legend(loc='upper right')
    plt.tight_layout(); plt.show()

    print(f'平均 {mean_v:>8.1f}   中央値 {median_v:>6.1f}   差 {mean_v - median_v:>8.1f}')


interact(compare_center, ceo_income=FloatSlider(min=500, max=10000, step=250, value=5000))

## 3. ばらつきの指標 — `ddof` の罠

$$
s^2 = \frac{1}{n}\sum (x_i - \bar{x})^2
\qquad\text{vs}\qquad
\hat{\sigma}^2 = \frac{1}{n-1}\sum (x_i - \bar{x})^2
$$

**NumPy の既定は `ddof=0`（$n$ で割る）**。母分散を推定したいなら `ddof=1` を明示します。

In [ ]:
print(f'標本分散 (ddof=0): {np.var(income):>12.1f}   ← NumPy の既定')
print(f'不偏分散 (ddof=1): {np.var(income, ddof=1):>12.1f}   ← 推定にはこちら')
print(f'標準偏差 (ddof=1): {np.std(income, ddof=1):>12.1f} 万円')
print(f'範囲             : {income.max() - income.min():>12.1f} 万円')
print(f'四分位範囲 IQR   : {stats.iqr(income):>12.1f} 万円   ← 外れ値に強い')
print(f'変動係数 CV      : {np.std(income, ddof=1) / np.mean(income):>12.3f}   ← 単位なし')

### なぜ `ddof=1` だと値が大きくなるのか

$\bar{x}$ は「そのデータに一番フィットする中心」なので、
$\bar{x}$ からの散らばりは**本当の $\mu$ からの散らばりより小さめ**に出ます。
その過小評価を補正するため、分母を小さくして値を持ち上げているのでした。

シミュレーションで確かめます。真の分散が分かっている分布から何度もサンプリングして、
どちらが真値に当たるか見てみましょう。

In [ ]:
TRUE_VAR: float = 4.0          # 真の分散 σ² = 4 (σ = 2)
N: int = 5                      # 小標本ほど差が出る
N_TRIALS: int = 20000

sim_rng = np.random.default_rng(0)
samples = sim_rng.normal(0.0, np.sqrt(TRUE_VAR), size=(N_TRIALS, N))  # shape: (20000, 5)

var_biased = samples.var(axis=1, ddof=0).mean()    # n で割った版の平均
var_unbiased = samples.var(axis=1, ddof=1).mean()  # n-1 で割った版の平均

print(f'真の分散 σ²          : {TRUE_VAR:.4f}')
print(f'ddof=0 の平均        : {var_biased:.4f}   ← 小さめに偏る (過小評価)')
print(f'ddof=1 の平均        : {var_unbiased:.4f}   ← ほぼ真値 (不偏)')
print()
print(f'ddof=0 は理論上 σ²×(n-1)/n = {TRUE_VAR * (N-1)/N:.4f} に偏る → 実測と一致')

## 4. 五数要約と箱ひげ図

$$
\min,\quad Q_1,\quad Q_2(\text{中央値}),\quad Q_3,\quad \max
$$

In [ ]:
q1, q2, q3 = np.percentile(income, [25, 50, 75])
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f'最小 {income.min():>6.0f}')
print(f'Q1   {q1:>6.1f}')
print(f'中央 {q2:>6.1f}')
print(f'Q3   {q3:>6.1f}')
print(f'最大 {income.max():>6.0f}')
print(f'\nIQR = {iqr:.1f}')
print(f'外れ値の目安: {lower_fence:.1f} 未満 または {upper_fence:.1f} 超')
print(f'→ 外れ値と判定される値: {income[(income < lower_fence) | (income > upper_fence)]}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].boxplot(income, vert=False, widths=0.6)
axes[0].set_title('箱ひげ図 (外れ値が点で飛び出す)')
axes[0].set_xlabel('年収 (万円)')
axes[1].boxplot(income[income < 1000], vert=False, widths=0.6)
axes[1].set_title('社長を除いた 9 人')
axes[1].set_xlabel('年収 (万円)')
plt.tight_layout(); plt.show()

> ⚠️ **1.5×IQR は作図上の慣習であって、「異常データの定義」ではありません。**
> 外れ値を見つけたら、捨てる前に「なぜその値なのか」を必ず調べてください。
> 入力ミスなのか、本当に起きた重要な現象なのかで、対応は正反対になります。

## 5. 歪度と尖度 — 分布の形を数値にする

- **歪度 (skewness)**: 左右の非対称性。$>0$ なら右に裾が長い
- **尖度 (kurtosis)**: 尖り具合と裾の重さ。SciPy の既定は**超過尖度**(正規分布 = 0)

In [ ]:
shape_rng = np.random.default_rng(1)
n = 5000
datasets = {
    '正規分布 (左右対称)': shape_rng.normal(0, 1, n),
    '対数正規 (右に裾)':   shape_rng.lognormal(0, 0.8, n),
    '一様分布 (裾が軽い)': shape_rng.uniform(-2, 2, n),
    't分布 df=3 (裾が重い)': shape_rng.standard_t(3, n),
}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (name, d) in zip(axes, datasets.items()):
    ax.hist(d, bins=60, density=True, alpha=0.75)
    ax.set_title(f'{name}\n歪度 {stats.skew(d):.2f} / 尖度 {stats.kurtosis(d):.2f}', fontsize=9)
    ax.set_yticks([])
plt.tight_layout(); plt.show()

print('歪度 > 0 → 平均 > 中央値 になっているか確認:')
for name, d in datasets.items():
    print(f'  {name:<22} 歪度 {stats.skew(d):>6.2f}  平均 {d.mean():>6.2f}  中央値 {np.median(d):>6.2f}')

## 6. ⚠️ アンスコムの四重奏 — 要約統計量を信じすぎない

1973 年に統計学者フランク・アンスコムが作った 4 つのデータセットです。

**平均・分散・相関係数・回帰直線が、4 つともほぼ完全に一致します。**

まず数値を見てください。

In [ ]:
# アンスコムの四重奏 (Anscombe 1973) — 有名な固定データ
x_common = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], dtype=float)
anscombe = {
    'I':   (x_common,
            np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    'II':  (x_common,
            np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    'III': (x_common,
            np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    'IV':  (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], dtype=float),
            np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}

print(f'{"組":<5}{"xの平均":>9}{"yの平均":>9}{"xの分散":>9}{"yの分散":>9}{"相関":>8}{"回帰直線":>22}')
print('-' * 66)
for name, (x, y) in anscombe.items():
    slope, intercept = np.polyfit(x, y, 1)
    r = np.corrcoef(x, y)[0, 1]
    print(f'{name:<5}{x.mean():>9.2f}{y.mean():>9.2f}{x.var(ddof=1):>9.2f}'
          f'{y.var(ddof=1):>9.2f}{r:>8.3f}      y={intercept:.2f}+{slope:.3f}x')

### 数値はそっくり。では図は?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7.5))
for ax, (name, (x, y)) in zip(axes.ravel(), anscombe.items()):
    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(2, 20, 50)
    ax.scatter(x, y, s=60, zorder=3)
    ax.plot(xs, intercept + slope * xs, 'r--', lw=1.5, label='同じ回帰直線')
    ax.set_title(f'データ {name}')
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('アンスコムの四重奏: 統計量は同じ、形はまったく違う', y=1.00)
plt.tight_layout(); plt.show()

print('I   : きれいな直線関係 → 回帰が妥当')
print('II  : 放物線 → 直線をあてはめるのが間違い')
print('III : 直線 + 外れ値 1 つ → 外れ値が回帰直線を引っ張っている')
print('IV  : x がほぼ一定 + 1 点 → その 1 点だけで直線が決まっている')
print()
print('★ 教訓: 要約統計量は図の代わりにならない。必ず描くこと。')

## 7. 欠損値の扱い

`np.mean` は nan があると nan を返します。これは「気づかせるための親切」です。

In [ ]:
data_with_nan: np.ndarray = np.array([10.0, 20.0, np.nan, 40.0, 50.0])  # shape: (5,)

print(f'np.mean    : {np.mean(data_with_nan)}      ← 1 つでも nan があれば nan')
print(f'np.nanmean : {np.nanmean(data_with_nan)}     ← nan を無視して計算')
print(f'欠損の個数 : {np.isnan(data_with_nan).sum()}')
print(f'欠損の割合 : {np.isnan(data_with_nan).mean():.1%}')
print()
print('⚠️ 「無視できるか」は中身次第。')
print('   アンケートで年収欄だけ空欄が多いなら、それはランダムな欠損ではない。')
print('   高所得者・低所得者が答えなかった可能性があり、無視すると結果が歪む。')

## まとめ

- **平均は外れ値に弱い**。裾が長い分布では中央値を見る
- **`np.var` の既定は `ddof=0`**。母分散を推定するなら `ddof=1`
- **五数要約と箱ひげ図**で全体像をつかむ。1.5×IQR は慣習であって定義ではない
- **歪度 > 0 なら平均 > 中央値**
- **アンスコムの四重奏**: 要約統計量が同じでも中身は別物。必ず図を描く
- **欠損は「消す」前に「なぜ欠けたか」を考える**

次は、この $\bar{x}$ から見えない母集団 $\mu$ を推し量る話へ。

→ 次: [`06_estimation.ipynb`](06_estimation.ipynb)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`04_bayes.ipynb`](04_bayes.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`06_estimation.ipynb`](06_estimation.ipynb) |